In [3]:
import pandas as pd
import numpy as np
import requests, os, time
from sqlalchemy import create_engine

In [2]:
# Hub Airport Coordinates (Latitude, Longitude)
AIRPORTS = {
    'DXB': (25.2532, 55.3657),  # Dubai International
    'AUH': (24.4330, 54.6511),  # Abu Dhabi International
    'DOH': (25.2731, 51.6081),  # Hamad International (Doha)
    'RUH': (24.9576, 46.6988)   # King Khalid International (Riyadh)
}

def haversine_km(lat1, lon1, lat2, lon2):
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    km = 6371.0 * c  # Earth radius in kilometers
    return km

In [ ]:
# Database connection parameters
DB_USER = "postgres"
DB_PASS = os.getenv("DB_PASS")  # Replace with your actual password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "gulf_aviation"

# Create Database Engine
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
while True:
    try:
        baseline_url = "https://opensky-network.org/api/states/all?lamin=15.0&lamax=32.0&lomin=34.0&lomax=60.0"
        response = requests.get(baseline_url)
        
        response.raise_for_status()  # Raise an error for bad responses
        # 1. Store the JSON response
        json_data = response.json()

        # 2. Define the exact column schema according to OpenSky API specs
        columns = [
            "icao24", "callsign", "origin_country", "time_position", 
            "last_contact", "longitude", "latitude", "baro_altitude", 
            "on_ground", "velocity", "true_track", "vertical_rate", 
            "sensors", "geo_altitude", "squawk", "spi", "position_source"
        ]
        
        # 3. Create a clean, flat DataFrame
        df_flights = pd.DataFrame(json_data['states'], columns=columns)

        # 4. Add the snapshot timestamp as a metadata column
        df_flights['snapshot_time'] = json_data['time']

        # Clean up callsign whitespace
        df_flights['callsign'] = df_flights['callsign'].str.strip()

        # Convert to numeric and convert meters to feet
        df_flights['altitude_ft'] = pd.to_numeric(df_flights['baro_altitude'], errors='coerce') * 3.28084 

        # Convert Velocity from m/s to knots
        df_flights['velocity_knots'] = pd.to_numeric(df_flights['velocity'], errors='coerce') * 1.94384

        # Convert Vertical Rate from m/s to feet/min
        df_flights['vertical_rate_fpm'] = pd.to_numeric(df_flights['vertical_rate'], errors='coerce') * 196.850394

        # Convert Unix timestamps to UTC standard datetime
        df_flights['time_position'] = pd.to_datetime(df_flights['time_position'], unit='s', utc=True)
        df_flights['snapshot_time'] = pd.to_datetime(df_flights['snapshot_time'], unit='s', utc=True)

        # 5. Classify flight status based on vertical rate and on_ground status
        df_flights['flight_status'] = np.select(
            [df_flights['on_ground'] == True , df_flights['vertical_rate_fpm'] > 300, df_flights['vertical_rate_fpm'] < -300],
            ['Ground', 'Climbing', 'Descending'],
            default='Cruising'
        )

        # Compute distance (in km) to each hub
        for hub_code, (hub_lat, hub_lon) in AIRPORTS.items():
            df_flights[f'dist_{hub_code}_km'] = haversine_km(
                df_flights['latitude'], df_flights['longitude'], 
                hub_lat, hub_lon
            )

        # Extract column names created for distances
        hub_dist_cols = [f'dist_{code}_km' for code in AIRPORTS.keys()]

        # Find the minimum distance and assign the corresponding airport code
        df_flights['dist_to_hub_km'] = df_flights[hub_dist_cols].min(axis=1)
        df_flights['nearest_hub'] = df_flights[hub_dist_cols].idxmin(axis=1).str.replace('dist_', '').str.replace('_km', '')

        # Clean up the DataFrame by dropping intermediate distance columns and renaming for clarity. Also, drop unnecessary columns.
        df_flights.drop(columns=hub_dist_cols, inplace=True)
        df_flights.drop(columns=['last_contact', 'time_position','baro_altitude', "velocity", "true_track", "vertical_rate", 
                                "sensors", "geo_altitude", "squawk", "spi", "position_source"], inplace=True)
        
        # Writing the processed DataFrame to a CSV file
        if os.path.exists('../data/processed/live_flights.csv'):
            df_flights.to_csv('../data/processed/live_flights.csv', mode='a', header=False, index=False)
        else:
            df_flights.to_csv('../data/processed/live_flights.csv', index=False)
        print("CSV file created successfully")

        # Append clean snapshot into PostgreSQL
        df_flights.to_sql('fact_flight_states', con=engine, if_exists='append', index=False)
        print("Flight snapshot successfully written to PostgreSQL!")
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        time.sleep(20)  # Wait for 20 seconds before the next request